# Домашнє завдання: Внесення оновлень в БД і робота з транзакціями

Це ДЗ передбачене під виконання на локальній машині. Виконання з Google Colab буде суттєво ускладнене.

## Підготовка
1. Переконайтесь, що у вас встановлены необхідні бібліотеки:
   ```bash
   pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv
   ```

2. Створіть файл `.env` з параметрами підключення до бази даних classicmodels. Базу даних ви можете отримати через

  - docker-контейнер згідно існтрукції в [документі](https://www.notion.so/hannapylieva/Docker-1eb94835849480c9b2e7f5dc22ee4df9), також відео інструкції присутні на платформі - уроки "MySQL бази, клієнт для роботи з БД, Docker і ChatGPT для запитів" та "Як встановити Docker для роботи з базами даних без терміналу"
  - або встановивши локально цю БД - для цього перегляньте урок "Опціонально. Встановлення MySQL та  БД Сlassicmodels локально".
  
  Приклад `.env` файлу ми створювали в лекції. Ось його обовʼязкове наповнення:
    ```
    DB_HOST=your_host
    DB_PORT=3306 або 3307 - той, який Ви налаштували
    DB_USER=your_username
    DB_PASSWORD=your_password
    DB_NAME=classicmodels
    ```
  Якщо ви створили цей файл під час перегляду лекції - **новий створювати не треба**. Замініть лише назву БД, або пропишіть назву в коді створення підключення (замість отримання назви цільової БД зі змінних оточення). Але переконайтесь, що до `.env` файл лежить в тій самій папці, що і цей ноутбук.

  **УВАГА!** НЕ копіюйте скрит для **створення** `.env` файлу. В лекції він наводиться для прикладу. І давалось пояснення, що в реальних проєктах ми НІКОЛИ не пишемо доступи до бази в коді. Копіювання скрипта для створення `.env` файлу сюди в ДЗ буде вважатись грубою помилкою і ми зніматимемо бали.

3. Налаштуйте підключення через SQLAlchemy до БД за прикладом в лекції.

Рекомендую вивести (відобразити) змінну engine після створення. Вона має бути не None! Якщо None - значить у Вас не підтягнулись налаштування з .env файла.

Ви також можете налаштувати параметри підключення до БД без .env файла, просто прописавши текстом в відповідних місцях. Це - не рекомендований підхід.


## Завдання

### Завдання 1: Оновлення інформації про клієнта (2 бали)

**Створіть функцію для оновлення контактної інформації клієнта за його номером** з наступними можливостями:
- Оновлення телефону клієнта
- Оновлення email (якщо поле існує в таблиці)

Опціонально, якщо вам хочеться більше практики:
- Логування змін в окрему таблицю

Використайте підхід з параметризованими запитами через `text()` та `UPDATE` оператор. Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом
```
  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
```

Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.



In [2]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [6]:
!pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv --quiet

In [33]:
def create_connection():
    load_dotenv()

    host = os.getenv('DB_HOST')
    port = os.getenv('DB_PORT', '3306')
    user = os.getenv('DB_USER')
    password = os.getenv('DB_PASSWORD')
    database = os.getenv('DB_NAME')

    connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"

    engine = create_engine(
        connection_string,
        pool_size=1,
        max_overflow=20,
        pool_pre_ping=True,
        echo=False
    )

    return engine

engine = create_connection()

In [36]:
def update_customer_contact(engine, customer_number, phone=None, email=None):

    if not phone and not email:
        print("❌ Немає даних для оновлення")
        return False

    # Отримуємо колонки таблиці
    columns_query = text("""
        SELECT COLUMN_NAME, DATA_TYPE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = 'customers'
    """)

    with engine.connect() as conn:
        columns = [row[0] for row in conn.execute(columns_query).fetchall()]

    # Перевіряємо клієнта
    check_query = text("""
        SELECT customerName
        FROM customers
        WHERE customerNumber = :customerNumber
    """)

    with engine.connect() as conn:
        customer = conn.execute(
            check_query,
            {"customerNumber": customer_number}
        ).fetchone()

        if not customer:
            print(f"❌ Клієнт {customer_number} не знайдений")
            return False

        print(f"👤 Оновлюємо клієнта: {customer[0]} (ID: {customer_number})")

    fields = []
    params = {"customerNumber": customer_number}

    if phone:
        fields.append("phone = :phone")
        params["phone"] = phone

    if email:
        if "email" in columns:
            fields.append("email = :email")
            params["email"] = email
        else:
            print("⚠️ Колонка email відсутня — пропускаємо")

    if not fields:
        print("❌ Немає валідних полів для оновлення")
        return False

    update_query = text(f"""
        UPDATE customers
        SET {", ".join(fields)}
        WHERE customerNumber = :customerNumber
    """)

    with engine.connect() as conn:
        with conn.begin():
            result = conn.execute(update_query, params)

    print(f"✅ Оновлено рядків: {result.rowcount}")
    return True


In [44]:
update_customer_contact(
    engine,
    customer_number=112,
    phone="+1 555 123 2222",
    email="new_email@example.com"
)

👤 Оновлюємо клієнта: Signal Gift Stores (ID: 112)
⚠️ Колонка email відсутня — пропускаємо
✅ Оновлено рядків: 1


True

### Завдання 2: Створення нового замовлення з транзакцією (5 балів)

**Реалізуйте процес створення нового замовлення** з наступними кроками в одній транзакції:
- Створення запису в таблиці `orders`
- Додавання товарних позицій в `orderdetails`
- Перевірка наявності товарів на складі
- Зменшення кількості товарів на складі

Запустіть процес з тестовими даними і продемонструйте через SELECT, що процес успішно відпрацював і були виконані необхідні операції.




In [45]:
def create_order_minimal(engine):
    with engine.connect() as conn:
        with conn.begin(): 
            try:
                result = conn.execute(text("""
                    INSERT INTO orders (orderDate, requiredDate, status, customerNumber)
                    VALUES (CURDATE(), CURDATE() + INTERVAL 7 DAY, 'In Process', 103)
                """))

                order_number = result.lastrowid
                print("Замовлення створено:", order_number)

                # Додаємо товар
                conn.execute(text("""
                    INSERT INTO orderdetails
                    (orderNumber, productCode, quantityOrdered, priceEach, orderLineNumber)
                    VALUES (:orderNumber, 'S10_1678', 1, 95.70, 1)
                """), {"orderNumber": order_number})

                # Зменшуємо склад
                conn.execute(text("""
                    UPDATE products
                    SET quantityInStock = quantityInStock - 1
                    WHERE productCode = 'S10_1678'
                """))

                print("✅ Транзакція успішна")
                return order_number

            except Exception as e:
                print("❌ Помилка:", e)
                return None


In [46]:
order_id = create_order_minimal(engine)




❌ Помилка: (pymysql.err.OperationalError) (1364, "Field 'orderNumber' doesn't have a default value")
[SQL: 
                    INSERT INTO orders (orderDate, requiredDate, status, customerNumber)
                    VALUES (CURDATE(), CURDATE() + INTERVAL 7 DAY, 'In Process', 103)
                ]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
